In [ ]:

-- clean up
DROP VIEW IF EXISTS global_temp.g_result;
DROP VIEW IF EXISTS global_temp.g_segmented_rows;
DROP VIEW IF EXISTS global_temp.g_universe;

-- g_universe
CREATE OR REPLACE GLOBAL TEMP VIEW g_universe AS
WITH universe AS (
    SELECT
        P.fiscal_year_period_date,
        C.SalesOfficeCode,
        C.CustomerGroupCode,
        P.branch_number,
        CASE WHEN M.PackagingMaterialCode = 'RPCK' THEN 'RPCK' ELSE 'CASE' END AS PackagingMaterialCode,
        C.PrimeVendorDealNumber,
        P.buying_group_code,
        C.CustomerNumber,
        C.RgnPostAcuteSaleGrpNumber,
        P.revenue_amt,
        P.sample_sales_amt,
        P.embrdry_sales_amt,
        P.distributor_rebate_accrual_amt,
        P.group_rebate_accrual_amt,
        P.customer_incentive_rebate_amt,
        P.corporate_rebate_accrual_amt,
        P.invoice_line_number,
        P.sales_order_number
    FROM profitability_analysis_fact P
    JOIN MaterialMaster M ON M.MaterialNumber = P.material_number
    JOIN CustomerMaster C ON C.CustomerNumber = P.customer_number
    WHERE P.fiscal_year_date IN ('2025','2024')
      AND C.SalesOfficeCode <> 'IC'
      AND P.branch_number NOT IN ('DIR','B52','RDM','PDM','B04','LRD','S02','TEM','SAMP','SHR','K44')
      AND P.product_division_code NOT IN ('12','95','83','11','98','13','01')
      AND P.vendor_rebate_accrual_amt = '0.00'
      AND P.billing_type_code = 'F2'
      AND P.material_category_group_code = 'NORM'
      AND P.cost_element_code <> '0000410998'
      AND P.ref_procedure_code NOT IN ('XIPAC','XIPST')
)
SELECT * FROM universe
;

-- g_segmented_rows
CREATE OR REPLACE GLOBAL TEMP VIEW g_segmented_rows AS
WITH segmented_rows AS (
    SELECT
        u.*,
        CASE
            WHEN u.CustomerGroupCode = 'HU' THEN 'CGC_HU'
            WHEN u.CustomerGroupCode = 'UM' THEN 'CGC_UM'
            WHEN u.CustomerGroupCode = 'OP' THEN 'CGC_OP'
            WHEN u.CustomerNumber IN (
                 'CUST_KA1','CUST_KA2','CUST_KA3','CUST_KA4',
                 'CUST_KA5','CUST_KA6','CUST_KA7','CUST_KA8','CUST_KA9'
            ) THEN 'HEALTH_SYSTEM_K'
            WHEN u.buying_group_code IN ('GRP_A','GRP_B')
                 AND u.CustomerNumber IN ('CUSTOMER_A','CUSTOMER_B','CUSTOMER_C',
                                          'CUSTOMER_D','CUSTOMER_E','CUSTOMER_F') THEN 'HEALTH_SYSTEM_A'
            WHEN u.buying_group_code IN ('GRP_C','GRP_D')
                 AND u.RgnPostAcuteSaleGrpNumber = 'SEG_CODE_A' THEN 'HEALTH_SYSTEM_B'
            WHEN u.PrimeVendorDealNumber LIKE '%XXXXXX' THEN 'HEALTH_SYSTEM_C'
            WHEN u.buying_group_code = 'GRP_E'  THEN 'HEALTH_SYSTEM_D'
            WHEN u.buying_group_code = 'GRP_F' THEN 'HEALTH_SYSTEM_E'
            ELSE 'General'
        END AS Segment
    FROM global_temp.g_universe u
)
SELECT * FROM segmented_rows
;

-- g_result
CREATE OR REPLACE GLOBAL TEMP VIEW g_result AS
SELECT
    fiscal_year_period_date,
    SalesOfficeCode,
    CustomerGroupCode,
    branch_number,
    PackagingMaterialCode,
    Segment,
    SUM(revenue_amt + sample_sales_amt + embrdry_sales_amt) AS GrossSales,
    SUM(revenue_amt + sample_sales_amt + embrdry_sales_amt
        + distributor_rebate_accrual_amt + group_rebate_accrual_amt
        + customer_incentive_rebate_amt + corporate_rebate_accrual_amt) AS NetSales,
    COUNT(invoice_line_number) AS Lines
FROM global_temp.g_segmented_rows
GROUP BY
    fiscal_year_period_date, SalesOfficeCode, CustomerGroupCode,
    branch_number, PackagingMaterialCode, Segment
;

In [ ]:

SHOW VIEWS IN global_temp;

SELECT COUNT(*) AS g_universe_rows       FROM global_temp.g_universe;
SELECT COUNT(*) AS g_segmented_rows_rows FROM global_temp.g_segmented_rows;
SELECT COUNT(*) AS g_result_groups       FROM global_temp.g_result;

In [ ]:

WITH base AS (
  SELECT
    SUM(revenue_amt + sample_sales_amt + embrdry_sales_amt) AS GrossSales,
    SUM(revenue_amt + sample_sales_amt + embrdry_sales_amt
        + distributor_rebate_accrual_amt + group_rebate_accrual_amt
        + customer_incentive_rebate_amt + corporate_rebate_accrual_amt) AS NetSales,
    COUNT(invoice_line_number) AS Lines
  FROM global_temp.g_universe
),
seg AS (
  SELECT
    SUM(GrossSales) AS GrossSales,
    SUM(NetSales)   AS NetSales,
    SUM(Lines)      AS Lines
  FROM global_temp.g_result
)
SELECT
  b.GrossSales AS Base_Gross,
  s.GrossSales AS Seg_Gross,
  (s.GrossSales - b.GrossSales) AS Diff_Gross,
  b.NetSales   AS Base_Net,
  s.NetSales   AS Seg_Net,
  (s.NetSales - b.NetSales)     AS Diff_Net,
  b.Lines      AS Base_Lines,
  s.Lines      AS Seg_Lines,
  (s.Lines - b.Lines)           AS Diff_Lines
FROM base b CROSS JOIN seg s
;

In [ ]:

WITH base AS (
  SELECT
    fiscal_year_period_date,
    SalesOfficeCode,
    CustomerGroupCode,
    branch_number,
    PackagingMaterialCode,
    SUM(revenue_amt + sample_sales_amt + embrdry_sales_amt) AS GrossSales,
    SUM(revenue_amt + sample_sales_amt + embrdry_sales_amt
        + distributor_rebate_accrual_amt + group_rebate_accrual_amt
        + customer_incentive_rebate_amt + corporate_rebate_accrual_amt) AS NetSales,
    COUNT(invoice_line_number) AS Lines
  FROM global_temp.g_universe
  GROUP BY fiscal_year_period_date, SalesOfficeCode, CustomerGroupCode, branch_number, PackagingMaterialCode
),
seg AS (
  SELECT
    fiscal_year_period_date,
    SalesOfficeCode,
    CustomerGroupCode,
    branch_number,
    PackagingMaterialCode,
    SUM(GrossSales) AS GrossSales,
    SUM(NetSales)   AS NetSales,
    SUM(Lines)      AS Lines
  FROM global_temp.g_result
  GROUP BY fiscal_year_period_date, SalesOfficeCode, CustomerGroupCode, branch_number, PackagingMaterialCode
)
SELECT
  COALESCE(b.fiscal_year_period_date, s.fiscal_year_period_date) AS fiscal_year_period_date,
  COALESCE(b.SalesOfficeCode, s.SalesOfficeCode) AS SalesOfficeCode,
  COALESCE(b.CustomerGroupCode, s.CustomerGroupCode) AS CustomerGroupCode,
  COALESCE(b.branch_number, s.branch_number) AS branch_number,
  COALESCE(b.PackagingMaterialCode, s.PackagingMaterialCode) AS PackagingMaterialCode,
  COALESCE(s.GrossSales,0) - COALESCE(b.GrossSales,0) AS Diff_Gross,
  COALESCE(s.NetSales,0)   - COALESCE(b.NetSales,0)   AS Diff_Net,
  COALESCE(s.Lines,0)      - COALESCE(b.Lines,0)      AS Diff_Lines
FROM base b
FULL OUTER JOIN seg s
  ON  b.fiscal_year_period_date = s.fiscal_year_period_date
  AND b.SalesOfficeCode         = s.SalesOfficeCode
  AND b.CustomerGroupCode       = s.CustomerGroupCode
  AND b.branch_number           = s.branch_number
  AND b.PackagingMaterialCode   = s.PackagingMaterialCode
WHERE
    COALESCE(s.GrossSales,0) <> COALESCE(b.GrossSales,0)
 OR COALESCE(s.NetSales,0)   <> COALESCE(b.NetSales,0)
 OR COALESCE(s.Lines,0)      <> COALESCE(b.Lines,0)
ORDER BY 1,2,3,4,5
;

In [ ]:

-- NULL 세그먼트 유무
SELECT
  SUM(CASE WHEN Segment IS NULL THEN 1 ELSE 0 END) AS Null_Segment_Rows,
  COUNT(*) AS Total_Rows
FROM global_temp.g_segmented_rows
;

-- 규칙 중복 적중 탐지
WITH flags AS (
  SELECT
    *,
    CASE WHEN CustomerGroupCode = 'HU' THEN 1 ELSE 0 END AS f_HU,
    CASE WHEN CustomerGroupCode = 'UM' THEN 1 ELSE 0 END AS f_UM,
    CASE WHEN CustomerGroupCode = 'OP' THEN 1 ELSE 0 END AS f_OP,
    CASE WHEN CustomerNumber IN (
         'CUST_KA1','CUST_KA2','CUST_KA3','CUST_KA4',
         'CUST_KA5','CUST_KA6','CUST_KA7','CUST_KA8','CUST_KA9') THEN 1 ELSE 0 END AS f_HSK,
    CASE WHEN buying_group_code IN ('GRP_A','GRP_B')
           AND CustomerNumber IN ('CUSTOMER_A','CUSTOMER_B','CUSTOMER_C','CUSTOMER_D','CUSTOMER_E','CUSTOMER_F')
         THEN 1 ELSE 0 END AS f_HSA,
    CASE WHEN buying_group_code IN ('GRP_C','GRP_D')
           AND RgnPostAcuteSaleGrpNumber = 'SEG_CODE_A'
         THEN 1 ELSE 0 END AS f_HSB,
    CASE WHEN PrimeVendorDealNumber LIKE '%XXXXXX' THEN 1 ELSE 0 END AS f_HSC,
    CASE WHEN buying_group_code = 'GRP_E'  THEN 1 ELSE 0 END AS f_HSD,
    CASE WHEN buying_group_code = 'GRP_F' THEN 1 ELSE 0 END AS f_HSE
  FROM global_temp.g_universe
)
SELECT
  SUM(CASE WHEN (f_HU+f_UM+f_OP+f_HSK+f_HSA+f_HSB+f_HSC+f_HSD+f_HSE) > 1 THEN 1 ELSE 0 END) AS Multi_Hit_Rows,
  COUNT(*) AS Total_Rows
FROM flags
;

In [ ]:

SELECT
  (SELECT COUNT(*) FROM global_temp.g_universe)       AS Universe_Rows,
  (SELECT COUNT(*) FROM global_temp.g_segmented_rows) AS Segmented_Rows,
  (SELECT COUNT(*) FROM global_temp.g_result)         AS Result_Groups
;

SELECT
  SUM(CASE WHEN cnt > 1 THEN 1 ELSE 0 END) AS Dup_OrderLine_Keys,
  COUNT(*) AS Distinct_OrderLine_Keys
FROM (
  SELECT sales_order_number, invoice_line_number, COUNT(*) AS cnt
  FROM global_temp.g_universe
  GROUP BY sales_order_number, invoice_line_number
) d
;

In [ ]:

SELECT
  Segment,
  COUNT(*) AS RowCnt,
  SUM(revenue_amt + sample_sales_amt + embrdry_sales_amt) AS GrossSales,
  SUM(revenue_amt + sample_sales_amt + embrdry_sales_amt
      + distributor_rebate_accrual_amt + group_rebate_accrual_amt
      + customer_incentive_rebate_amt + corporate_rebate_accrual_amt) AS NetSales
FROM global_temp.g_segmented_rows
GROUP BY Segment
ORDER BY RowCnt DESC
;

In [ ]:
-- 
-- AND P.fiscal_year_period_date BETWEEN '2025-01-01' AND '2025-01-31'

WITH u AS (
  SELECT
    P.sales_order_number, P.invoice_line_number
  FROM profitability_analysis_fact P
  WHERE P.fiscal_year_date IN ('2025','2024')
)
SELECT
  COUNT(*)                                        AS rows_total,
  COUNT(DISTINCT sales_order_number, invoice_line_number) AS rows_distinct,
  COUNT(*) - COUNT(DISTINCT sales_order_number, invoice_line_number) AS delta_rows
FROM u;

In [ ]:
-- 
WITH u AS (
  SELECT
    P.sales_order_number, P.invoice_line_number, P.material_number
  FROM profitability_analysis_fact P
  WHERE P.fiscal_year_date IN ('2025','2024')
)
SELECT
  sales_order_number, invoice_line_number,
  COUNT(*)                           AS cnt_rows,
  COUNT(DISTINCT material_number)    AS distinct_materials
FROM u
GROUP BY sales_order_number, invoice_line_number
HAVING COUNT(*) > 1
ORDER BY cnt_rows DESC
LIMIT 200;

In [ ]:
-- CustomerMaster 중복 키
SELECT CustomerNumber, COUNT(*) AS cnt
FROM CustomerMaster
GROUP BY CustomerNumber
HAVING COUNT(*) > 1
LIMIT 50;

-- MaterialMaster 중복 키
SELECT MaterialNumber, COUNT(*) AS cnt
FROM MaterialMaster
GROUP BY MaterialNumber
HAVING COUNT(*) > 1
LIMIT 50;

In [ ]:
WITH u AS (
  SELECT
    P.fiscal_year_period_date,
    P.sales_order_number, P.invoice_line_number,
    P.revenue_amt, P.sample_sales_amt, P.embrdry_sales_amt,
    P.distributor_rebate_accrual_amt, P.group_rebate_accrual_amt,
    P.customer_incentive_rebate_amt, P.corporate_rebate_accrual_amt,
    C.SalesOfficeCode, C.CustomerGroupCode, P.branch_number,
    CASE WHEN M.PackagingMaterialCode='RPCK' THEN 'RPCK' ELSE 'CASE' END AS PackagingMaterialCode
  FROM profitability_analysis_fact P
  JOIN CustomerMaster C ON C.CustomerNumber = P.customer_number
  JOIN MaterialMaster  M ON M.MaterialNumber  = P.material_number
  WHERE P.fiscal_year_date IN ('2025','2024')
)
,base AS (
  SELECT
    fiscal_year_period_date, SalesOfficeCode, CustomerGroupCode, branch_number, PackagingMaterialCode,
    SUM(revenue_amt + sample_sales_amt + embrdry_sales_amt) AS Gross_noSeg,
    SUM(revenue_amt + sample_sales_amt + embrdry_sales_amt
        + distributor_rebate_accrual_amt + group_rebate_accrual_amt
        + customer_incentive_rebate_amt + corporate_rebate_accrual_amt) AS Net_noSeg,
    COUNT(invoice_line_number) AS Lines_raw,
    COUNT(DISTINCT sales_order_number, invoice_line_number) AS Lines_distinct
  FROM u
  GROUP BY 1,2,3,4,5
)
SELECT
  SUM(Gross_noSeg) AS Gross_total,
  SUM(Net_noSeg)   AS Net_total,
  SUM(Lines_raw)   AS Lines_raw_total,
  SUM(Lines_distinct) AS Lines_distinct_total,
  SUM(Lines_raw) - SUM(Lines_distinct) AS Lines_delta_total
FROM base;

In [ ]:
-- 사실/차원 각각의 고객번호 길이 비교
SELECT MIN(LENGTH(CAST(customer_number AS STRING))) AS p_min_len,
       MAX(LENGTH(CAST(customer_number AS STRING))) AS p_max_len
FROM profitability_analysis_fact;

SELECT MIN(LENGTH(CustomerNumber)) AS c_min_len,
       MAX(LENGTH(CustomerNumber)) AS c_max_len
FROM CustomerMaster;